In [1]:
# Load the libraries and locate the project
import hashlib
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

current = Path.cwd()
project = current.parent if current.name == "notebooks" else current

In [2]:
# Load training flows from the prepared CSV
data_file = project / "notebooks" / "prepared_network_traffic.csv"
if not data_file.is_file():
    raise FileNotFoundError("Run data_preprocessing.ipynb first")
prepared = pd.read_csv(data_file)
required = {"Label", "time_group", "split"}
if not required.issubset(prepared.columns):
    raise ValueError("Prepared CSV is missing Label, time_group, or split")
feature_names = [name for name in prepared if name not in required]
if len(feature_names) != 69 or set(prepared["split"]) != {"train", "validation", "test"}:
    raise ValueError("Prepared CSV needs 69 features and three data splits")
train = prepared["split"].eq("train")
X_train = prepared.loc[train, feature_names]
y_train = prepared.loc[train, "Label"].astype(int)
train_groups = prepared.loc[train, "time_group"]
print("Training flows:", len(X_train), "Features:", len(feature_names))

Training flows: 139968 Features: 69


In [3]:
# Choose how to compare settings
# Each two-minute group stays together in a validation fold.
validation = StratifiedGroupKFold(
    n_splits=2, shuffle=True, random_state=7
)
print("Two training-only validation folds; DDoS F1 chooses the best setting")

Two training-only validation folds; DDoS F1 chooses the best setting


In [4]:
# Compare two Logistic Regression settings
logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
logistic_search = GridSearchCV(
    logistic, {"model__C": [0.1, 1.0]},
    scoring="f1", cv=validation, n_jobs=1, refit=False,
)
logistic_search.fit(X_train, y_train, groups=train_groups)
display(pd.DataFrame(logistic_search.cv_results_)[
    ["params", "mean_test_score", "std_test_score"]
])
print("Best setting:", logistic_search.best_params_)

,params,mean_test_score,std_test_score
0,{'model__C': 0.1},0.992494,0.002961
1,{'model__C': 1.0},0.997909,0.000883


Best setting: {'model__C': 1.0}


In [5]:
# Compare two Random Forest settings
forest = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=100, n_jobs=2, random_state=42
    )),
])
forest_search = GridSearchCV(
    forest, {"model__max_depth": [16, None]},
    scoring="f1", cv=validation, n_jobs=1, refit=False,
)
forest_search.fit(X_train, y_train, groups=train_groups)
display(pd.DataFrame(forest_search.cv_results_)[
    ["params", "mean_test_score", "std_test_score"]
])
print("Best setting:", forest_search.best_params_)

,params,mean_test_score,std_test_score
0,{'model__max_depth': 16},0.999621,0.000186
1,{'model__max_depth': None},0.999615,0.000192


Best setting: {'model__max_depth': 16}


In [6]:
# Save tuning settings as a one-row CSV
with data_file.open("rb") as file:
    prepared_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
settings = {
    "logistic_regression_C": float(logistic_search.best_params_["model__C"]),
    "random_forest_max_depth": forest_search.best_params_["model__max_depth"],
    "logistic_cv_f1": float(logistic_search.best_score_),
    "forest_cv_f1": float(forest_search.best_score_),
    "prepared_csv_sha256": prepared_sha256,
}
settings_path = project / "notebooks" / "tuning_results.csv"
pd.DataFrame([settings]).to_csv(settings_path, index=False)
print("Saved:", settings_path.resolve())

Saved: C:\Users\joy\Downloads\Jupyter\FlowGuard-69-Release\notebooks\tuning_results.csv
